In [207]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

In [208]:
df = pd.read_csv("sku_22423_weekly_demand.csv")
df.head()

,StockCode,InvoiceDate,Quantity
0,22423,2010-12-05,456
1,22423,2010-12-12,648
2,22423,2010-12-19,363
3,22423,2010-12-26,87
4,22423,2011-01-09,237


In [209]:
df["Quantity"].describe()

count     53.000000
mean     233.471698
std      108.610915
min       72.000000
25%      149.000000
50%      214.000000
75%      300.000000
max      648.000000
Name: Quantity, dtype: float64

In [210]:
mean_demand = df["Quantity"].mean()
std_demand = df["Quantity"].std()

# Self-note: Average weekly demand ≈ 233 units; Weekly demand typically fluctuates by about 109 units around that average

print(mean_demand)
print(std_demand)

233.47169811320754
108.61091526346995


### Scenario 1: Balanced Costs

In [211]:
selling_price = 10
purchase_cost = 6
salvage_value = 2

Cu = selling_price - purchase_cost
Co = purchase_cost - salvage_value

print("Cu =", Cu)
print("Co =", Co)

Cu = 4
Co = 4


## Monte Carlo Simulation

In [212]:
np.random.seed(42)

simulations = 1000

simulated_demand = np.random.normal(mean_demand, std_demand, n_simulations)
# Generate many possible future weeks of demand based on the historical pattern

simulated_demand = np.maximum(simulated_demand, 0)

In [213]:
simulated_demand[:10] 

# first 10 simulated weekly demand scenarios generated by Monte Carlo

array([287.4202769 , 218.45468581, 303.81774304, 398.88936479,
       208.04008577, 208.04186892, 404.9914474 , 316.82348645,
       182.48165536, 292.39964103])

In [214]:
Q = 233 # optimal Q for scenario 1 

#### Profit for Every Simulated Week

In [215]:
profits = []

for demand in simulated_demand:
    sales_unit = min(Q, demand)

    leftover = max(Q - demand, 0)

    profit = (sales_unit * selling_price - Q * purchase_cost + leftover * salvage_value)

    profits.append(profit)

In [216]:
profits = pd.Series(profits)

profits.describe()

count    1000.000000
mean      606.615392
std       467.965382
min      -932.000000
25%       373.090579
50%       932.000000
75%       932.000000
max       932.000000
dtype: float64

*The Monte Carlo simulation highlights the impact of demand uncertainty on inventory performance. While the Newsvendor model recommends ordering 233 units, simulated profits ranged from -932 to 932 units depending on actual demand. This demonstrates that even an optimal inventory policy cannot eliminate the risks associated with uncertain demand.*

*Self-note: The worst-case profit (-932) occurs when demand = 0. The best-case profit (932) occurs when demand is = 233, allowing all ordered units to be sold. The average simulated profit was approximately 607 units.*

In [217]:
stockout_rate = (simulated_demand > Q).mean()

leftover_rate = (simulated_demand < Q).mean()

print(stockout_rate)
print(leftover_rate)

0.511
0.489


Out of 1000 simulated weeks:
- 511 weeks → Demand > 214 → Stockout
- 489 weeks → Demand < 214 → Leftover 

The stockout and leftover frequencies were close to 50-50, suggesting that the order quantity of 233 units achieved a relatively balanced trade-off between understocking and overstocking under the simulated demand conditions.

### Scenario 2: Stockouts are Expensive

In [218]:
Q = 288

selling_price = 15
purchase_cost = 6
salvage_value = 2

#### Profit for Every Simulated Week

In [219]:
profits = []

for demand in simulated_demand:
    sales_unit = min(Q, demand)

    leftover = max(Q - demand, 0)

    profit = (sales_unit * selling_price - Q * purchase_cost + leftover * salvage_value)

    profits.append(profit)

In [220]:
profits = pd.Series(profits)

profits.describe()

count    1000.000000
mean     1636.988779
std       994.466395
min     -1152.000000
25%       968.772190
50%      1918.855070
75%      2592.000000
max      2592.000000
dtype: float64

In [221]:
stockout_rate = (simulated_demand > Q).mean()

leftover_rate = (simulated_demand < Q).mean()

print(stockout_rate)
print(leftover_rate)

0.305
0.695


*Monte Carlo simulation showed that profits ranged from -1152 to 2592 units, with an average profit of approximately 1636 units. Compared with the balanced-cost scenario, the higher order quantity (288 units) increased potential profit but also exposed the inventory policy to larger profit variability.*


*Self-note: Because stockouts were assumed to be more expensive, the model recommended a larger order quantity (288 units). When demand was high = the larger inventory allowed more sales and higher profit; when demand was low, higher the risk of excess inventory and potential losses*

### Scenario 3: Excess Inventory is Costly

In [222]:
Q = 187

selling_price = 8
purchase_cost = 6
salvage_value = 2

#### Profit for Every Simulated Week

In [223]:
profits = []

for demand in simulated_demand:
    sales_unit = min(Q, demand)

    leftover = max(Q - demand, 0)

    profit = (sales_unit * selling_price - Q * purchase_cost + leftover * salvage_value)

    profits.append(profit)

In [224]:
profits = pd.Series(profits)

profits.describe()

count    1000.000000
mean      243.091242
std       252.763567
min      -748.000000
25%       230.817934
50%       374.000000
75%       374.000000
max       374.000000
dtype: float64

*Monte Carlo simulation showed that profits ranged from -748 to 374 units, with an average profit of approximately 243 units. Compared with the other scenarios, the lower order quantity reduced the risk of excess inventory but also limited potential profits.*


*Self-note: Because excess inventory was assumed to be more costly, the model recommended a smaller order quantity (187 units). This reduces the amount of unsold inventory when demand is low, but increases the likelihood of stockouts and missed sales when demand is high.*

### Comparison

In [225]:
comparison = pd.DataFrame({
    "Scenario": [
        "Balanced Costs",
        "Stockouts Expensive",
        "Excess Inventory Costly"
    ],
    "Optimal Q": [233, 288, 187],
    "Average Profit": [607, 1636, 243]})

In [226]:
comparison

,Scenario,Optimal Q,Average Profit
0,Balanced Costs,233,607
1,Stockouts Expensive,288,1636
2,Excess Inventory Costly,187,243


## Conclusion

*This project applied inventory analytics techniques to the Online Retail dataset. After cleaning the data and identifying high-value products through ABC analysis, weekly demand patterns were analyzed for a selected SKU. The Newsvenfor model was then used to determine optimal inventory levels under different cost assumptions. Finally, Monte Carlo simulation evaluated inventory performance under demand uncertainty.*


*The results showed that higher stockout costs led to larger order quantities and higher profit potential, while higher excess inventory costs resulted in more conservative ordering decisions. The analysis highlights the trade-off between product availability and inventory risk, demonstrating how analytical models can support inventory decision-making under uncertainity.*